# Encoding & Scikit-learn Pipelines — Practical Notebook
### Dataset: IBM Telco Customer Churn (7,043 customers)

This notebook is the hands-on companion to **Encoding_Pipelines_Guide.docx**.
No theory dumps here — just code, run in order, with comments explaining
*why* each line exists. If a cell errors out, that is not bad luck, that is
the lesson. Read the error, fix it, move on.

**What you will build, step by step:**
1. Load & inspect the Telco Churn dataset
2. Identify nominal / ordinal / high-cardinality columns
3. Manually apply Label, One-Hot, Ordinal, Frequency & Target Encoding (to *see* what they do)
4. Build a `ColumnTransformer` that automates step 3 correctly
5. Wrap everything in a `Pipeline` (no data leakage, one `.fit()`, one `.predict()`)
6. Compare `Pipeline` vs `make_pipeline`
7. Tune the pipeline with `GridSearchCV`
8. Validate with `cross_val_score`
9. Save the final production-ready pipeline with `joblib`
10. **Exercises** — you do these yourself, no copy-pasting allowed


## 0. Setup — Imports & Loading the Data

In [1]:
# Core libraries
import pandas as pd
import numpy as np

# sklearn — preprocessing & encoding
from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
)
from sklearn.impute import SimpleImputer

# sklearn — composing & chaining
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline

# sklearn — modeling & evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
print("Libraries loaded.")

Libraries loaded.


In [2]:
# Load the IBM Telco Customer Churn dataset directly from IBM's public GitHub repo.
# If you don't have internet access, download the CSV manually and put it in the
# same folder as this notebook, then just set: url = "Telco-Customer-Churn.csv"

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Explore the Dataset & Identify Column Types

Before touching any encoder, you MUST know what you are encoding.
Never blindly One-Hot-Encode everything — that is how you end up with
300 columns and a model that overfits.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# TotalCharges is loaded as an 'object' (string) column, even though it's numeric.
# Classic real-world data problem: a few rows have blank strings ' ' instead of a number.
print(df['TotalCharges'].apply(type).value_counts())
print()
blank_rows = df[df['TotalCharges'].str.strip() == '']
print(f"Rows with blank TotalCharges: {len(blank_rows)}")
blank_rows[['customerID', 'tenure', 'TotalCharges']]

TotalCharges
<class 'str'>    7043
Name: count, dtype: int64

Rows with blank TotalCharges: 11


,customerID,tenure,TotalCharges
488,4472-LVYGI,0,
753,3115-CZMZD,0,
936,5709-LVOEQ,0,
1082,4367-NUYAO,0,
1340,1371-DWPAZ,0,
3331,7644-OMVMY,0,
3826,3213-VVOLG,0,
4380,2520-SGTTA,0,
5218,2923-ARZLG,0,
6670,4075-WKNIU,0,


In [5]:
# Fix it: convert to numeric, blanks become NaN (to be handled by an Imputer later —
# this is EXACTLY why real pipelines need a SimpleImputer step, not just an encoder).
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Missing values in TotalCharges now:", df['TotalCharges'].isna().sum())

Missing values in TotalCharges now: 11


In [6]:
# Drop the ID column — customerID is a unique identifier, not a predictive feature.
df = df.drop(columns=['customerID'])

# Separate columns by type (this is a decision YOU make, not something sklearn guesses)
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']

nominal_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
    'PaymentMethod'
]

ordinal_features = ['Contract']   # Month-to-month < One year < Two year -> TRUE order

target_col = 'Churn'

print("Numeric  :", numeric_features)
print("Nominal  :", nominal_features)
print("Ordinal  :", ordinal_features)
print("Target   :", target_col)

Numeric  : ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Nominal  : ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod']
Ordinal  : ['Contract']
Target   : Churn


In [7]:
# Always check cardinality (number of unique categories) BEFORE choosing an encoder.
for col in nominal_features + ordinal_features:
    print(f"{col:20s} -> {df[col].nunique()} unique values : {df[col].unique().tolist()}")

gender               -> 2 unique values : ['Female', 'Male']
Partner              -> 2 unique values : ['Yes', 'No']
Dependents           -> 2 unique values : ['No', 'Yes']
PhoneService         -> 2 unique values : ['No', 'Yes']
MultipleLines        -> 3 unique values : ['No phone service', 'No', 'Yes']
InternetService      -> 3 unique values : ['DSL', 'Fiber optic', 'No']
OnlineSecurity       -> 3 unique values : ['No', 'Yes', 'No internet service']
OnlineBackup         -> 3 unique values : ['Yes', 'No', 'No internet service']
DeviceProtection     -> 3 unique values : ['No', 'Yes', 'No internet service']
TechSupport          -> 3 unique values : ['No', 'Yes', 'No internet service']
StreamingTV          -> 3 unique values : ['No', 'Yes', 'No internet service']
StreamingMovies      -> 3 unique values : ['No', 'Yes', 'No internet service']
PaperlessBilling     -> 2 unique values : ['Yes', 'No']
PaymentMethod        -> 4 unique values : ['Electronic check', 'Mailed check', 'Bank transfer 

In [8]:
# Quick look at class balance in the target — important for model evaluation choices later.
df['Churn'].value_counts(normalize=True).rename('proportion')

,proportion
Churn,
No,0.73463
Yes,0.26537


## 2. Train / Test Split — BEFORE Any Encoding

This is the single most important line in this notebook. Split FIRST,
encode SECOND. If you encode (especially Target/Frequency encoding) on
the full dataset before splitting, you leak test-set information into
training — your accuracy will look great and be a lie.

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col].map({'No': 0, 'Yes': 1})   # Label Encoding the TARGET manually is fine

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

## 3. Manual Encoding Demonstrations (for learning only)

We will now manually apply each encoding type to a **copy** of the
training data, just to *see* what each one actually produces. In Section
4 we throw this manual code away and let `ColumnTransformer` do it
properly and safely.

### 3.1 Label Encoding

In [ ]:
demo = X_train.copy()

le = LabelEncoder()
demo['gender_label'] = le.fit_transform(demo['gender'])
print(dict(zip(le.classes_, le.transform(le.classes_))))
demo[['gender', 'gender_label']].head()

**Why this is risky here:** `gender` is *nominal* (Male/Female have no
order), but Label Encoding just gave it a fake numeric order (0 vs 1).
A Logistic Regression model would now assume 'Male' is mathematically
"greater than" 'Female' — meaningless. Label Encoding is fine for the
**target** column or for **tree-based models**, but risky for nominal
features fed into linear models.

### 3.2 One-Hot Encoding

In [ ]:
ohe_demo = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded_cols = ohe_demo.fit_transform(demo[['gender', 'InternetService']])
encoded_df = pd.DataFrame(
    encoded_cols,
    columns=ohe_demo.get_feature_names_out(['gender', 'InternetService']),
    index=demo.index
)
encoded_df.head()

### 3.3 Ordinal Encoding (with a real, meaningful order)

In [ ]:
contract_order = [['Month-to-month', 'One year', 'Two year']]
oe_demo = OrdinalEncoder(categories=contract_order)
demo['Contract_ordinal'] = oe_demo.fit_transform(demo[['Contract']])
demo[['Contract', 'Contract_ordinal']].drop_duplicates().sort_values('Contract_ordinal')

### 3.4 Frequency Encoding (pure pandas — zero leakage risk if fit on train only)

In [ ]:
freq_map = X_train['PaymentMethod'].value_counts(normalize=True)
demo['PaymentMethod_freq'] = demo['PaymentMethod'].map(freq_map)
demo[['PaymentMethod', 'PaymentMethod_freq']].drop_duplicates().sort_values('PaymentMethod_freq', ascending=False)

### 3.5 Target / Mean Encoding

**Danger zone.** Target Encoding uses the label (`y`), so it must be
fit on `X_train`/`y_train` ONLY — never on `X_test`. We implement it
here with plain pandas so you can see exactly what it computes; in a
real pipeline you would use `category_encoders.TargetEncoder` inside a
cross-validated fold to avoid leakage completely.

In [ ]:
train_demo = X_train.copy()
train_demo['Churn'] = y_train.values

target_means = train_demo.groupby('PaymentMethod')['Churn'].mean()
print(target_means.sort_values(ascending=False))

demo['PaymentMethod_target'] = demo['PaymentMethod'].map(target_means)
demo[['PaymentMethod', 'PaymentMethod_target']].drop_duplicates().sort_values('PaymentMethod_target', ascending=False)

**Notice:** Electronic check has the highest churn rate. This single
encoded number already tells the model "customers using this payment
method churn more" — much more informative than an arbitrary Label
Encoding integer, but it only works because we computed it from
`y_train`, not the full dataset.

## 4. Doing It Properly: `ColumnTransformer`

Forget the manual demo code above. This is how you do it for real: one
object, routes each column group to the right transformer, and is safe
to reuse identically on test data / production data.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # fixes the TotalCharges NaNs
    ('scaler', StandardScaler())
])

nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('nom', nominal_transformer, nominal_features),
    ('ord', ordinal_transformer, ordinal_features),
], remainder='drop')

print(preprocessor)

In [ ]:
# Quick sanity check: fit_transform on training data and look at the output shape.
X_train_transformed = preprocessor.fit_transform(X_train)
print("Transformed training shape:", X_train_transformed.shape)
print("(started with", X_train.shape[1], "raw columns)")

## 5. The Full `Pipeline`: Preprocessing + Model in ONE Object

This is the object you would actually deploy. One `.fit()`, one
`.predict()`, zero risk of forgetting a preprocessing step.

In [ ]:
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes'])

## 6. `make_pipeline` vs `Pipeline` — Side by Side

Same preprocessing, same model, same data. Only the step names differ.

In [ ]:
# Option A: Pipeline — you choose the names
pipe_a = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Option B: make_pipeline — names auto-generated
pipe_b = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))

pipe_a.fit(X_train, y_train)
pipe_b.fit(X_train, y_train)

print("Pipeline    step names :", list(pipe_a.named_steps.keys()))
print("make_pipeline step names:", list(pipe_b.named_steps.keys()))
print()
print("Identical predictions? ->", np.array_equal(pipe_a.predict(X_test), pipe_b.predict(X_test)))

Notice the identical predictions — `make_pipeline` is not a
different algorithm, it is just a shortcut that skips naming the steps
yourself. The trade-off shows up the moment you use `GridSearchCV`
below: you must reference the auto-generated name.

## 7. Hyperparameter Tuning with `GridSearchCV`

Because everything lives inside one `Pipeline`, `GridSearchCV` can tune
the model's hyperparameters while re-running the *entire* preprocessing
correctly on every cross-validation fold — this is impossible to do
safely without a Pipeline.

In [ ]:
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],          # note the double underscore: step_name__param
    'classifier__penalty': ['l2'],
}

grid_search = GridSearchCV(
    log_reg_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_.round(4))
print("Test accuracy with best model:", round(accuracy_score(y_test, grid_search.predict(X_test)), 4))

**With `make_pipeline`**, the exact same grid would need to use the
auto-generated name instead, e.g. `'logisticregression__C'` — this is
the practical cost of the convenience `make_pipeline` gives you.

## 8. Cross-Validation with `cross_val_score`

A quick, honest estimate of how the pipeline generalizes — again, the
Pipeline object guarantees preprocessing is refit correctly on every fold.

In [ ]:
cv_scores = cross_val_score(log_reg_pipeline, X_train, y_train, cv=5, scoring='accuracy')
print("Fold accuracies:", cv_scores.round(4))
print("Mean CV accuracy:", cv_scores.mean().round(4), " +/- ", cv_scores.std().round(4))

## 9. Tree-Based Model with Ordinal/Label-style Encoding

Random Forests don't need One-Hot Encoding the way linear models do —
they can split on raw integer-encoded categories just fine. This is why
your `ColumnTransformer` design should also depend on WHICH model you
are about to use, not encoding in isolation.

In [ ]:
rf_preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), nominal_features + ordinal_features),
])

rf_pipeline = make_pipeline(
    rf_preprocessor,
    RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)
)

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
print("Random Forest test accuracy:", round(accuracy_score(y_test, rf_pred), 4))

## 10. Save the Final Pipeline for Deployment

One file, everything included — preprocessing AND model. Whoever loads
this file in production does NOT need to know or re-implement any
encoding logic.

In [ ]:
joblib.dump(grid_search.best_estimator_, 'telco_churn_pipeline.pkl')
print("Saved: telco_churn_pipeline.pkl")

# Sanity check: reload and predict on one row
loaded_pipeline = joblib.load('telco_churn_pipeline.pkl')
sample = X_test.iloc[[0]]
print("Prediction on one sample row:", loaded_pipeline.predict(sample))

## 11. Exercises — Do These Yourself (No Peeking at Section 4-9)

If you copy-paste without typing these yourself, you will forget
everything in a week. Do them.

1. **Binary Encoding**: Install `category_encoders` and apply
   `ce.BinaryEncoder` to the `PaymentMethod` column. Compare the number
   of resulting columns to One-Hot Encoding.
2. **Add SMOTE or `class_weight='balanced'`** to `LogisticRegression`
   inside the pipeline and see if recall on the "Churn" class improves.
3. **Swap the model**: replace `LogisticRegression` with
   `GradientBoostingClassifier` inside the same `Pipeline` — you should
   only need to change ONE line.
4. **Break it on purpose**: remove the `SimpleImputer` step from
   `numeric_transformer` and re-run `.fit()`. Read the error message
   carefully — this is the exact bug a Pipeline is designed to prevent.
5. **Explain in your own words** (write it as a markdown cell below):
   why does `PaymentMethod_target` (Section 3.5) need to be computed
   using only `X_train`/`y_train`, never the full dataset?


In [ ]:
# Exercise 1 starter — uncomment and complete:
# import category_encoders as ce
# binary_enc = ce.BinaryEncoder(cols=['PaymentMethod'])
# binary_encoded = binary_enc.fit_transform(X_train['PaymentMethod'])
# print(binary_encoded.shape)
# binary_encoded.head()


---
### End of notebook.
Go back to `Encoding_Pipelines_Guide.docx` Section 5 (Cheat Sheet) and
make sure you can explain every row of that table without looking at
this notebook.